# Step 9 — Pipeline observability and run audit

This notebook verifies the externally visible observability outputs required by the assignment. It runs ETL and ELT, reads the `pipeline_runs` audit table, checks `dq_results`, and inserts a controlled failed-run example to demonstrate how failure history is represented.

The audit record captures run ID, DAG ID, timing, extracted rows, loaded rows, quarantined rows and status.

In [1]:
from pathlib import Path
import sys
from datetime import datetime, timezone
import duckdb

# Use the same pipeline modules as the Airflow DAGs.
sys.path.insert(0, str(Path('include').resolve()))
from nammamart_etl import run_etl
from nammamart_elt import run_elt
from observability import record_pipeline_run

etl_warehouse = Path('assignment/warehouse_samy_etl_step9.duckdb')
elt_warehouse = Path('assignment/warehouse_samy_elt_step9.duckdb')
etl_result = run_etl(warehouse_path=etl_warehouse, quarantine_dir='assignment/quarantine_samy_step9', run_id='step9-etl-success')
elt_result = run_elt(warehouse_path=elt_warehouse, run_id='step9-elt-success')
print(f"ETL fact rows: {etl_result['rows_loaded']:,}")
print(f"ELT passing staging rows: {elt_result['passing_orders']:,}")

ETL fact rows: 1,851
ELT passing staging rows: 2,318


In [2]:
# Inspect the audit history and quality scorecard generated by each run.
for label, warehouse in [('ETL', etl_warehouse), ('ELT', elt_warehouse)]:
    con = duckdb.connect(str(warehouse), read_only=True)
    print(f'--- {label} pipeline_runs ---')
    display(con.execute('SELECT * FROM pipeline_runs ORDER BY start_time').fetchdf())
    if label == 'ETL':
        print('--- ETL dq_results summary ---')
        display(con.execute("SELECT status, count(*) AS checks FROM dq_results GROUP BY status").fetchdf())
    con.close()

--- ETL pipeline_runs ---


,run_id,dag_id,start_time,end_time,rows_extracted,rows_loaded,rows_quarantined,status
0,step9-etl-success,nammamart_etl_samy,2026-10-11 02:35:38.029188,2026-10-11 02:35:38.193398,5372,1850,203,SUCCESS
1,step9-etl-controlled-failure,nammamart_etl_samy,2026-10-11 02:35:38.431940,2026-10-11 02:35:38.431940,0,0,0,FAILED
2,step9-etl-success,nammamart_etl_samy,2026-10-22 12:06:42.713074,2026-10-22 12:06:42.901903,5372,1851,201,SUCCESS
3,step9-etl-controlled-failure,nammamart_etl_samy,2026-10-22 12:06:43.351350,2026-10-22 12:06:43.351350,0,0,0,FAILED
4,step9-etl-success,nammamart_etl_samy,2026-10-22 12:07:09.084720,2026-10-22 12:07:09.271316,5372,1851,201,SUCCESS


--- ETL dq_results summary ---


,status,checks
0,FAIL,27
1,PASS,4


--- ELT pipeline_runs ---


,run_id,dag_id,start_time,end_time,rows_extracted,rows_loaded,rows_quarantined,status
0,step9-elt-success,nammamart_elt_samy,2026-10-11 02:35:38.379575,2026-10-11 02:35:38.379589,5212,2316,84,SUCCESS
1,step9-elt-success,nammamart_elt_samy,2026-10-22 12:06:43.208301,2026-10-22 12:06:43.208314,5212,2318,82,SUCCESS
2,step9-elt-success,nammamart_elt_samy,2026-10-22 12:07:09.452146,2026-10-22 12:07:09.452160,5212,2318,82,SUCCESS


In [3]:
# Record a controlled failure so the audit table demonstrates both states.
now = datetime.now(timezone.utc).isoformat()
record_pipeline_run(
    etl_warehouse,
    run_id='step9-etl-controlled-failure',
    dag_id='nammamart_etl_samy',
    start_time=now,
    end_time=now,
    rows_extracted=0,
    rows_loaded=0,
    rows_quarantined=0,
    status='FAILED',
)
con = duckdb.connect(str(etl_warehouse), read_only=True)
display(con.execute("SELECT status, count(*) AS runs FROM pipeline_runs GROUP BY status ORDER BY status").fetchdf())
con.close()
print('Observability test passed: success, failure and DQ history are queryable from DuckDB.')

,status,runs
0,FAILED,3
1,SUCCESS,3


Observability test passed: success, failure and DQ history are queryable from DuckDB.
